# 🎁 Python 装饰器 (Decorator) 简介

装饰器本质上是一个 **接收函数、返回函数** 的高阶函数，用来在不修改原函数代码的情况下，给函数添加额外功能。

目录：
1. 函数是一等公民
2. 最简单的装饰器
3. `@` 语法糖
4. 保留函数信息：`functools.wraps`
5. 带参数的装饰器
6. 实用场景示例

## 1. 函数是一等公民

在 Python 中，函数可以赋值给变量、作为参数传递、作为返回值——这是装饰器的基础。

In [1]:
def greet(name):
    return f"Hello, {name}!"

# 函数赋值给变量
say_hi = greet
print(say_hi("Alice"))

# 函数作为参数
def call_twice(func, arg):
    print(func(arg))
    print(func(arg))

call_twice(greet, "Bob")

Hello, Alice!
Hello, Bob!
Hello, Bob!


## 2. 最简单的装饰器

装饰器 = 一个函数，接收一个函数，返回一个新函数。

In [2]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print("⏩ 函数执行前")
        result = func(*args, **kwargs)
        print("⏪ 函数执行后")
        return result
    return wrapper

# 手动装饰
def say_hello():
    print("Hello!")

say_hello = my_decorator(say_hello)
say_hello()

⏩ 函数执行前
Hello!
⏪ 函数执行后


## 3. `@` 语法糖

`@decorator` 等价于 `func = decorator(func)`，写起来更简洁。

In [3]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print("⏩ 函数执行前")
        result = func(*args, **kwargs)
        print("⏪ 函数执行后")
        return result
    return wrapper

@my_decorator
def say_goodbye():
    print("Goodbye!")

say_goodbye()

⏩ 函数执行前
Goodbye!
⏪ 函数执行后


## 4. 保留函数信息：`functools.wraps`

装饰后，函数的 `__name__` 和 `__doc__` 会变成 wrapper 的。用 `@wraps` 修复。

In [4]:
from functools import wraps

def good_decorator(func):
    @wraps(func)  # ← 关键：保留原函数信息
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@good_decorator
def add(a, b):
    """两数相加"""
    return a + b

print(f"函数名: {add.__name__}")  # add（不是 wrapper）
print(f"文档:   {add.__doc__}")   # 两数相加

函数名: add
文档:   两数相加


## 5. 带参数的装饰器

需要再包一层函数：外层接收装饰器参数，内层是标准装饰器。

In [ ]:
from functools import wraps

def repeat(n):
    """让函数执行 n 次"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for i in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)
def say(msg):
    print(msg)

say("Hi!")

## 6. 实用场景示例

### 6.1 计时器装饰器

In [ ]:
import time
from functools import wraps

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"⏱️ {func.__name__} 耗时 {elapsed:.4f}s")
        return result
    return wrapper

@timer
def slow_function():
    time.sleep(0.5)
    return "done"

slow_function()

### 6.2 日志装饰器

In [ ]:
from functools import wraps

def log(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"📝 调用 {func.__name__}({args}, {kwargs})")
        result = func(*args, **kwargs)
        print(f"📝 返回 {result}")
        return result
    return wrapper

@log
def multiply(a, b):
    return a * b

multiply(3, 7)

### 6.3 缓存/记忆化（Python 内置）

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=128)
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print(f"fib(50) = {fibonacci(50)}")
print(f"缓存信息: {fibonacci.cache_info()}")

## 总结

| 概念 | 要点 |
|------|------|
| 基本结构 | 外层接收函数，内层 wrapper 包裹执行 |
| `@` 语法糖 | `@deco` 等价于 `func = deco(func)` |
| `@wraps` | 必加！保留原函数的 `__name__` 和 `__doc__` |
| 带参数 | 多套一层：`@deco(arg)` → 三层嵌套 |
| 常用场景 | 计时、日志、权限检查、缓存、重试 |

记住核心公式：
```python
def decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        # 前置逻辑
        result = func(*args, **kwargs)
        # 后置逻辑
        return result
    return wrapper
```